# Emmanuel's Face Detection and Recognition Notebook

**Purpose:** I detect one clear face, align it, create an SFace embedding, enrol consenting people, use separate validation images to choose the match threshold, then evaluate an untouched test set.

**Why a notebook:** I can run one small Python step at a time and inspect the face image and matching result. The dashboard still imports the reusable Python module; this notebook teaches and evaluates that same pipeline.

**Important:** SFace is already pretrained. My enrollment images build an identity gallery; they do not train SFace. Similarity is not the probability that two images show the same person. YuNet + SFace do not detect printed-photo or screen attacks.


## 1. Install or confirm the Python packages

I use OpenCV's face detector and face recognizer APIs plus NumPy. I install `opencv-contrib-python-headless` because it includes the YuNet and SFace APIs without requiring a desktop window. In Colab, run this cell once if those imports are missing.


In [ ]:
# If OpenCV or NumPy is missing in this notebook runtime, uncomment and run:
# %pip install "opencv-contrib-python-headless>=4.10,<5" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "remotezip>=0.12,<1"

from pathlib import Path
from hashlib import sha256
from urllib.request import urlopen
import os
import time
import zipfile

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)


## 2. Get the verified face models

I use YuNet to find facial landmarks and SFace to create 128-number embeddings. The notebook checks SHA-256 so an incomplete or unexpected model file is not used. I do not use FaceNet here: it is part of our literature review, while these OpenCV Zoo models are the pipeline available in this project.


In [ ]:
FACE_MODELS = {
    "face_detection_yunet_2023mar.onnx": {
        "url": "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
        "sha256": "8f2383e4dd3cfbb4553ea8718107fc0423210dc964f9f4280604804ed2552fa4",
    },
    "face_recognition_sface_2021dec.onnx": {
        "url": "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx",
        "sha256": "0ba9fbfa01b5270c96627c4ef784da859931e02f04419c829e83484087c34e79",
    },
}

# In this project notebook, the models folder is one level above this notebook.
# In Colab, it is normally beside the notebook, so I accept either location.
MODEL_NAMES = tuple(FACE_MODELS)
MODEL_DIR = Path("models")
for candidate_dir in (Path("models"), Path("../models")):
    if all((candidate_dir / name).is_file() for name in MODEL_NAMES):
        MODEL_DIR = candidate_dir
        break
MODEL_DIR.mkdir(exist_ok=True)

def file_sha256(path):
    digest = sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def get_verified_model(filename, details):
    path = MODEL_DIR / filename
    if not path.exists():
        print("Downloading", filename)
        with urlopen(details["url"], timeout=120) as response:
            path.write_bytes(response.read())
    actual = file_sha256(path)
    if actual != details["sha256"]:
        raise ValueError(f"Checksum did not match for {filename}. Do not use this file.")
    print("Verified:", filename)
    return path

yunet_path = get_verified_model("face_detection_yunet_2023mar.onnx", FACE_MODELS["face_detection_yunet_2023mar.onnx"])
sface_path = get_verified_model("face_recognition_sface_2021dec.onnx", FACE_MODELS["face_recognition_sface_2021dec.onnx"])


## 3. Create the detector and recognizer once

I load each model once and reuse it. This avoids paying the model-loading cost for every image and helps keep processing quicker.


In [ ]:
DETECTION_THRESHOLD = 0.70  # YuNet face-detection score, not identity similarity.
MAX_IMAGE_SIDE = 640           # Keep image detail while bounding inference cost.

face_detector = cv2.FaceDetectorYN.create(
    str(yunet_path), "", (320, 320),
    score_threshold=DETECTION_THRESHOLD,
    nms_threshold=0.30,
)
face_recognizer = cv2.FaceRecognizerSF.create(str(sface_path), "")
print("YuNet and SFace are ready.")


## 4. Detect, align, and represent one face

I require exactly one detectable face in each image. This prevents me from silently enrolling or matching the wrong person in a group photo. I align facial landmarks to standardize the crop before SFace compares it. I use shadow enhancement only for YuNet detection; SFace receives the original image so enrollment and testing use the same appearance pipeline. If the camera captured no facial detail, software enhancement cannot recreate it.


In [ ]:
def improve_dark_detection(image_bgr):
    # Lift shadow detail for detection only; keep the original for SFace.
    ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    luminance = ycrcb[:, :, 0]
    if float(np.mean(luminance)) < 100:
        ycrcb[:, :, 0] = cv2.createCLAHE(
            clipLimit=2.0, tileGridSize=(8, 8)
        ).apply(luminance)
        return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)
    return image_bgr

def read_bgr(image_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"I could not read this image: {image_path}")
    return image

def make_embedding(image_bgr):
    # Return a unit-length SFace vector and the aligned face image.
    if image_bgr is None or image_bgr.size == 0 or image_bgr.ndim != 3:
        raise ValueError("I need a nonempty colour image.")
    if image_bgr.shape[2] != 3 or image_bgr.dtype != np.uint8:
        raise ValueError("I need a uint8 BGR colour image.")

    height, width = image_bgr.shape[:2]
    scale = min(1.0, MAX_IMAGE_SIDE / max(height, width))
    resized = cv2.resize(
        image_bgr,
        (max(1, round(width * scale)), max(1, round(height * scale))),
        interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR,
    )

    detect_image = improve_dark_detection(resized)
    face_detector.setInputSize((resized.shape[1], resized.shape[0]))
    _, faces = face_detector.detect(detect_image)
    if faces is None or len(faces) != 1:
        found = 0 if faces is None else len(faces)
        raise ValueError(f"I expected exactly one clear face, but found {found}.")

    aligned = face_recognizer.alignCrop(resized, faces[0])
    vector = face_recognizer.feature(aligned).flatten().astype(np.float32)
    length = float(np.linalg.norm(vector))
    if vector.shape != (128,) or not np.isfinite(vector).all() or length == 0:
        raise ValueError("SFace returned an invalid embedding.")
    return vector / length, aligned, faces[0]

def show_face(image_bgr, title="Face image"):
    plt.figure(figsize=(5, 4))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


## 5. Prepare separate enrollment, validation, and test images

I keep the three sets separate. Enrollment builds the gallery, validation chooses the threshold, and test is used once for final evaluation. Each file must be a distinct capture; do not split augmented copies of one photo. Known people need images in all three sets. People in `Unknown` must never be enrolled. Aim for at least 20 test captures. An exact 60/20/20 split needs 100 images to put 20 in test; 80 images gives 48/16/16.

For a repeatable practice run, I download a small subset from Microsoft's **DigiFace-1M**. It contains generated identities, not consenting volunteers, and is licensed for non-commercial research. This lets me practice the split and code without collecting real people's faces. It cannot establish real-camera performance, lighting robustness, or spoof resistance. For the project evaluation, I must still test consented camera/video captures.

```text
face_images/
  enroll/
    Ada Lovelace/   (at least 3 separate real captures)
      front.jpg
      slight_turn.jpg
  validation/
    Ada Lovelace/   (separate captures of an enrolled person)
      validation_1.jpg
    Unknown/        (people absent from the gallery)
      unknown_1.jpg
  test/
    Ada Lovelace/   (separate captures, not used for enrolment or validation)
      held_out_1.jpg
    Unknown/        (people absent from the gallery)
      held_out_unknown_1.jpg
```

Source: [Microsoft DigiFace-1M](https://github.com/microsoft/DigiFace1M). I read and accepted its non-commercial research terms. The notebook downloads only selected files into the ignored `private_data/` folder. I can set `USE_DIGIFACE_SAMPLE = False` and provide my own consented images instead. I keep images and embeddings off GitHub. I tune the threshold using validation only; I do not tune it after looking at test results.

Dataset paper: Bae et al., *DigiFace-1M: 1 Million Digital Face Images for Face Recognition*, WACV 2023.


In [ ]:
USE_DIGIFACE_SAMPLE = True
DATA_DIR = Path("private_data/digiface1m_subset/face_images")

if USE_DIGIFACE_SAMPLE:
    # I fetch 140 selected PNGs by byte-range from the official 2.9 GB archive.
    # remotezip reads only the selected image files instead of downloading the full archive.
    from remotezip import RemoteZip

    DIGIFACE_URL = (
        "https://facesyntheticspubwedata.z6.web.core.windows.net/wacv-2023/"
        "subjects_0-1999_72_imgs.zip"
    )
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    if not any(DATA_DIR.rglob("*.png")):
        with RemoteZip(DIGIFACE_URL, support_suffix_range=False) as archive:
            # Known identities: 3 gallery images, 1 validation image, 1 test image each.
            for identity_id in range(20):
                person_name = f"Synthetic_{identity_id:04d}"
                split_paths = {
                    "enroll": range(0, 3),
                    "validation": range(3, 4),
                    "test": range(4, 5),
                }
                for split_name, image_numbers in split_paths.items():
                    person_folder = DATA_DIR / split_name / person_name
                    person_folder.mkdir(parents=True, exist_ok=True)
                    for image_number in image_numbers:
                        member = f"{identity_id}/{image_number}.png"
                        (person_folder / f"capture_{image_number}.png").write_bytes(
                            archive.read(member)
                        )

            # Unknown validation and test identities are disjoint from each other.
            for split_name, identity_range in (
                ("validation", range(20, 40)),
                ("test", range(40, 60)),
            ):
                unknown_folder = DATA_DIR / split_name / "Unknown"
                unknown_folder.mkdir(parents=True, exist_ok=True)
                for identity_id in identity_range:
                    member = f"{identity_id}/0.png"
                    (unknown_folder / f"synthetic_unknown_{identity_id:04d}.png").write_bytes(
                        archive.read(member)
                    )
    print("Small synthetic DigiFace-1M practice set is ready at:", DATA_DIR)
else:
    # In Colab, upload a ZIP containing face_images/enroll, validation, and test.
    try:
        from google.colab import files
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    DATA_DIR = Path("face_images")
    if IN_COLAB:
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise ValueError("Upload exactly one ZIP containing the face_images folders.")
        DATA_DIR.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_names[0]) as archive:
            for item in archive.infolist():
                destination = (DATA_DIR / item.filename).resolve()
                if not destination.is_relative_to(DATA_DIR.resolve()):
                    raise ValueError("The ZIP contains a path outside the data folder.")
            archive.extractall(DATA_DIR)
        if (DATA_DIR / "face_images").is_dir():
            DATA_DIR = DATA_DIR / "face_images"

enroll_dir = DATA_DIR / "enroll"
validation_dir = DATA_DIR / "validation"
test_dir = DATA_DIR / "test"
missing_folders = [
    name for name, folder in (
        ("enroll", enroll_dir),
        ("validation", validation_dir),
        ("test", test_dir),
    )
    if not folder.is_dir()
]
if missing_folders:
    raise FileNotFoundError(
        "I need separate enroll, validation, and test folders. Missing: "
        + ", ".join(missing_folders)
    )
print("Enrollment people:", [p.name for p in enroll_dir.iterdir() if p.is_dir()])
print("Validation labels:", [p.name for p in validation_dir.iterdir() if p.is_dir()])
print("Test labels:", [p.name for p in test_dir.iterdir() if p.is_dir()])


## 6. Enrol people from separate real captures

I make one normalized template per person by averaging that person's normalized capture embeddings and normalizing the average again. This uses several real views without pretending that synthetic copies add new evidence.


In [ ]:
def image_files(folder):
    extensions = {".jpg", ".jpeg", ".png", ".bmp"}
    return sorted(path for path in folder.rglob("*") if path.suffix.lower() in extensions)

def normalized_average(vectors):
    average = np.mean(np.stack(vectors), axis=0)
    length = float(np.linalg.norm(average))
    if not np.isfinite(average).all() or length == 0:
        raise ValueError("I could not make a valid average template.")
    return (average / length).astype(np.float32)

gallery = {}
enrollment_counts = {}

for person_folder in sorted(enroll_dir.iterdir()):
    if not person_folder.is_dir():
        continue
    captures = image_files(person_folder)
    if not captures:
        print("Skipping empty folder:", person_folder.name)
        continue
    vectors = []
    for path in captures:
        try:
            vector, aligned_face, _ = make_embedding(read_bgr(path))
            vectors.append(vector)
        except ValueError as error:
            print("Skipping unclear enrollment image:", path.name, "—", error)
    if len(vectors) < 2:
        print("Skipping identity with fewer than 2 clear enrollment images:", person_folder.name)
        continue
    gallery[person_folder.name] = normalized_average(vectors)
    enrollment_counts[person_folder.name] = f"{len(vectors)}/{len(captures)} clear captures"

if not gallery:
    raise ValueError("I need at least one enrolled person with a clear face image.")
print("Enrolled capture counts:", enrollment_counts)
print("Profiles are held in this notebook memory; I have not saved the face images.")


## 7. Choose the threshold using validation images only

I normalize the embedding because SFace compares vector direction; after normalization, a dot product gives cosine similarity quickly. I keep the ambiguity margin provisional and use only validation images to choose the similarity threshold. Test images are not used to tune settings. I never describe similarity as a probability.


In [ ]:
SIMILARITY_THRESHOLD = 0.40  # Replaced below using validation images only.
AMBIGUITY_MARGIN = 0.05       # Provisional gap required over the second-best identity.

def compare_embedding(query, references):
    scores = {
        name: float(np.clip(np.dot(query, reference), -1.0, 1.0))
        for name, reference in references.items()
    }
    ranked = sorted(scores.items(), key=lambda pair: (-pair[1], pair[0]))
    candidate, best_score = ranked[0]
    second_score = ranked[1][1] if len(ranked) > 1 else None
    score_gap = None if second_score is None else best_score - second_score

    if best_score < SIMILARITY_THRESHOLD:
        status, accepted_name = "UNKNOWN", None
        reason = "No enrolled reference passed the similarity threshold."
    elif score_gap is not None and score_gap < AMBIGUITY_MARGIN:
        status, accepted_name = "UNCERTAIN", None
        reason = "The best two enrolled references are too close."
    else:
        status, accepted_name = "KNOWN", candidate
        reason = "The score passed the provisional decision rules."

    return {
        "status": status,
        "accepted_name": accepted_name,
        "closest_candidate": candidate,
        "cosine_similarity": best_score,
        "second_best_gap": score_gap,
        "reason": reason,
        "all_scores": scores,
    }

def recognize_image(image_path):
    vector, aligned_face, _ = make_embedding(read_bgr(image_path))
    return compare_embedding(vector, gallery), aligned_face

def tune_threshold(validation_folder):
    validation_rows = []
    for label_folder in sorted(validation_folder.iterdir()):
        if not label_folder.is_dir():
            continue
        expected = label_folder.name
        is_unknown = expected.casefold() == "unknown"
        for path in image_files(label_folder):
            try:
                vector, _, _ = make_embedding(read_bgr(path))
                scores = {
                    name: float(np.clip(np.dot(vector, template), -1.0, 1.0))
                    for name, template in gallery.items()
                }
                ranked_scores = sorted(scores.items(), key=lambda pair: (-pair[1], pair[0]))
                best_name, best_score = ranked_scores[0]
                second_score = ranked_scores[1][1] if len(ranked_scores) > 1 else None
                gap = None if second_score is None else best_score - second_score
                validation_rows.append((expected, is_unknown, best_name, best_score, gap))
            except ValueError as error:
                print("Skipping unclear validation image:", path.name, "—", error)

    if not validation_rows:
        raise ValueError("I need clear validation images before tuning the threshold.")
    if not any(row[1] for row in validation_rows) or not any(not row[1] for row in validation_rows):
        raise ValueError("Validation needs both known identities and an Unknown folder.")

    candidates = sorted({0.0, 1.0, *(row[3] for row in validation_rows)})
    known_rows = [row for row in validation_rows if not row[1]]
    unknown_rows = [row for row in validation_rows if row[1]]
    best_threshold = None
    best_accuracy = -1.0
    for threshold in candidates:
        known_correct = sum(
            best_score >= threshold
            and (gap is None or gap >= AMBIGUITY_MARGIN)
            and best_name == expected
            for expected, _, best_name, best_score, gap in known_rows
        )
        unknown_correct = sum(
            best_score < threshold
            for _, _, _, best_score, _ in unknown_rows
        )
        # Give known and unknown examples equal weight even if folder counts differ.
        accuracy = 0.5 * (
            known_correct / len(known_rows)
            + unknown_correct / len(unknown_rows)
        )
        if accuracy >= best_accuracy:
            best_accuracy = accuracy
            best_threshold = threshold
    return best_threshold, best_accuracy, len(validation_rows)

SIMILARITY_THRESHOLD, validation_accuracy, validation_count = tune_threshold(validation_dir)
print("Validation images used:", validation_count)
print("Chosen cosine-similarity threshold:", round(SIMILARITY_THRESHOLD, 3))
print("Validation accuracy at chosen threshold:", round(validation_accuracy, 3))
print("This is threshold tuning, not the final test result.")


## 8. Evaluate the untouched test images and inspect results

I run each test image once after freezing the validation threshold. I put people missing from the gallery in `test/Unknown`. I report the number of test images, correct decisions, unknown false accepts, and known false rejects. Fewer than 20 images triggers a reminder to collect more; this is not a claim of general accuracy.


In [ ]:
results = []
aligned_examples = []

for label_folder in sorted(test_dir.iterdir()):
    if not label_folder.is_dir():
        continue
    expected = label_folder.name
    is_unknown = expected.casefold() == "unknown"
    for path in image_files(label_folder):
        try:
            decision, aligned = recognize_image(path)
            if is_unknown:
                correct = decision["status"] == "UNKNOWN"
            else:
                correct = decision["accepted_name"] == expected
            results.append({
                "image": path.name,
                "expected_folder": expected,
                "decision": decision["status"],
                "accepted_name": decision["accepted_name"],
                "closest_candidate": decision["closest_candidate"],
                "cosine_similarity": round(decision["cosine_similarity"], 3),
                "correct_for_this_label": correct,
                "reason": decision["reason"],
            })
            if len(aligned_examples) < 6:
                aligned_examples.append((path.name, aligned))
        except ValueError as error:
            results.append({
                "image": path.name,
                "expected_folder": expected,
                "decision": "FACE_UNCLEAR",
                "accepted_name": None,
                "closest_candidate": None,
                "cosine_similarity": None,
                "correct_for_this_label": False,
                "reason": str(error),
            })

if not results:
    print("No test images found. Add separate images under face_images/test/ and rerun this cell.")
else:
    for row in results:
        print(row)
    print("\nImages evaluated:", len(results))
    print("Correct for these supplied labels:", sum(r["correct_for_this_label"] for r in results), "/", len(results))
    unknown_rows = [row for row in results if row["expected_folder"].casefold() == "unknown"]
    known_rows = [row for row in results if row["expected_folder"].casefold() != "unknown"]
    false_accepts = sum(row["decision"] == "KNOWN" for row in unknown_rows)
    false_rejects = sum(row["accepted_name"] != row["expected_folder"] for row in known_rows)
    print("Unknown false accepts:", false_accepts, "/", len(unknown_rows))
    print("Known false rejects or wrong identities:", false_rejects, "/", len(known_rows))
    if len(results) < 20:
        print("Reminder: collect at least 20 distinct test images for the planned evaluation.")
    print("This result describes only these supplied images and scenarios.")
    if aligned_examples:
        figure, axes = plt.subplots(1, len(aligned_examples), figsize=(3 * len(aligned_examples), 3))
        if len(aligned_examples) == 1:
            axes = [axes]
        for axis, (name, face) in zip(axes, aligned_examples):
            axis.imshow(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))
            axis.set_title(name)
            axis.axis("off")
        plt.tight_layout()
        plt.show()


## 9. Test a recorded video or a local camera

This cell reads a recorded clip, or a camera attached to the computer running Jupyter, and checks faces every second frame. I save an annotated MP4 plus average, median, and 95th-percentile face-processing time. On Colab I upload a video file; a Colab runtime cannot directly open the camera attached to my laptop. To try DroidCam locally, I set `VIDEO_SOURCE` to its camera index (often not `0`). This tests face recognition on frames; person tracking, restricted-zone crossings, and alert latency still need the full application test.


In [ ]:
VIDEO_SOURCE = Path("private_data/add_test_video.mp4")  # Or set 0 / the DroidCam camera index in local Jupyter.
VIDEO_OUTPUT = Path("private_data/video_results/face_video_review.mp4")
FRAME_STEP = 2       # I process every second frame to reduce the work per second.
MAX_VIDEO_FRAMES = 300  # Short repeatable test; increase for longer clips.

def identify_faces_in_frame(frame_bgr):
    height, width = frame_bgr.shape[:2]
    scale = min(1.0, MAX_IMAGE_SIDE / max(height, width))
    frame = cv2.resize(
        frame_bgr,
        (max(1, round(width * scale)), max(1, round(height * scale))),
        interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR,
    )
    detect_image = improve_dark_detection(frame)
    face_detector.setInputSize((frame.shape[1], frame.shape[0]))
    _, faces = face_detector.detect(detect_image)
    observations = []
    if faces is None:
        return frame, observations

    for face in faces:
        aligned = face_recognizer.alignCrop(frame, face)
        vector = face_recognizer.feature(aligned).flatten().astype(np.float32)
        length = float(np.linalg.norm(vector))
        if length == 0 or not np.isfinite(vector).all():
            continue
        decision = compare_embedding(vector / length, gallery)
        x, y, box_width, box_height = [int(value) for value in face[:4]]
        if decision["status"] == "KNOWN":
            label = decision["accepted_name"]
            colour = (40, 180, 40)
        else:
            label = decision["status"]
            colour = (30, 40, 220)
        cv2.rectangle(frame, (x, y), (x + box_width, y + box_height), colour, 2)
        cv2.putText(
            frame, label, (x, max(20, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2,
        )
        observations.append({
            "status": decision["status"],
            "name": decision["accepted_name"],
            "similarity": round(decision["cosine_similarity"], 3),
        })
    return frame, observations

def review_video(source, output_path, frame_step=2, max_frames=300):
    capture_source = str(source) if isinstance(source, Path) else source
    capture = cv2.VideoCapture(capture_source)
    if not capture.isOpened():
        raise ValueError("I could not open the video or camera. Check its path or camera index.")

    source_fps = capture.get(cv2.CAP_PROP_FPS)
    output_fps = source_fps if source_fps and source_fps > 0 else 20.0
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = None
    processing_seconds = []
    total_frames = 0
    total_faces = 0
    video_started = time.perf_counter()
    try:
        while total_frames < max_frames:
            read_ok, frame = capture.read()
            if not read_ok:
                break
            height, width = frame.shape[:2]
            scale = min(1.0, MAX_IMAGE_SIDE / max(height, width))
            frame = cv2.resize(
                frame,
                (max(1, round(width * scale)), max(1, round(height * scale))),
                interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR,
            )
            if writer is None:
                height, width = frame.shape[:2]
                writer = cv2.VideoWriter(
                    str(output_path), cv2.VideoWriter_fourcc(*"mp4v"),
                    output_fps, (width, height),
                )
                if not writer.isOpened():
                    raise ValueError("I could not create the output MP4 file.")

            if total_frames % frame_step == 0:
                started = time.perf_counter()
                frame, observations = identify_faces_in_frame(frame)
                processing_seconds.append(time.perf_counter() - started)
                total_faces += len(observations)
            writer.write(frame)
            total_frames += 1
    finally:
        capture.release()
        if writer is not None:
            writer.release()
    wall_seconds = time.perf_counter() - video_started

    if total_frames == 0:
        raise ValueError("I did not read any frames from this source.")
    if not processing_seconds:
        raise ValueError("I did not process any frames; increase MAX_VIDEO_FRAMES.")
    latencies_ms = np.array(processing_seconds) * 1000
    print("Frames written:", total_frames)
    print("Frames analyzed:", len(processing_seconds))
    print("Face observations:", total_faces)
    print("End-to-end processing rate:", round(total_frames / wall_seconds, 2), "video frames/second")
    print("Face-analysis rate:", round(len(processing_seconds) / sum(processing_seconds), 2), "analyzed frames/second")
    print("Face-processing latency p50 / p95:", round(float(np.percentile(latencies_ms, 50)), 1), "/", round(float(np.percentile(latencies_ms, 95)), 1), "ms")
    print("Annotated video saved to:", output_path)

if isinstance(VIDEO_SOURCE, Path) and not VIDEO_SOURCE.is_file():
    print("Put a test clip at", VIDEO_SOURCE, "or change VIDEO_SOURCE to your video path/camera index.")
else:
    review_video(VIDEO_SOURCE, VIDEO_OUTPUT, FRAME_STEP, MAX_VIDEO_FRAMES)


## 10. What this experiment does and does not establish

- A failed face detection is **FACE_UNCLEAR**, not an unknown identity.
- An unknown decision means no enrolled template passed the configured threshold. It does not mean the person is dangerous.
- The score is cosine similarity, not a percent chance.
- SFace is pretrained; the enrollment set builds the identity gallery, it does not retrain SFace.
- This DigiFace result is a synthetic-image practice run, not evidence of real people or camera performance.
- Use separate real captures for enrollment, validation, and test. Keep the threshold fixed after validation and before test.
- This notebook has no MiniFASNet anti-spoof model, MediaPipe FaceMesh, blink test, or liveness decision. Printed photos, screens, twins, caps, distance, angle, and darkness need separate test scenarios before any performance claim.
- The video cell checks face recognition on sampled frames. It does not implement person tracking, restricted-zone crossings, deduplicated alerts, or end-to-end alert latency; those must be measured in the full application.
- Report participant count, distinct capture count, false accepts, false rejects, uncertain/unclear faces, and per-scenario results.
